<a href="https://colab.research.google.com/github/Reece-Challinor/60daysto/blob/dev/Copy_of_Mistral_2x24B_MOE_Power_Magistral_Devstral_Reasoning_Ultimate_44B_i1_GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get install -y build-essential


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [1]:
!apt-get update

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,005 kB]
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Pa

In [4]:
!python -m pip install --upgrade --verbose pip setuptools wheel packaging build

SyntaxError: invalid syntax (ipython-input-1663952312.py, line 1)

In [7]:
%%bash

set -euo pipefail

# Verbose pip + sane defaults
export PIP_PROGRESS_BAR=on
export PIP_DEFAULT_TIMEOUT=120
export PIP_ROOT_USER_ACTION=ignore

# 1) Upgrade pip/build toolchain (verbose)

# 2) Detect CUDA version (if any)
CUDA_VER="$(
python - <<'PY'
import re, subprocess
try:
    out = subprocess.check_output(["nvidia-smi", "-q"], stderr=subprocess.STDOUT).decode()
    m = re.search(r"CUDA Version\s*:\s*([\d.]+)", out)
    print(m.group(1) if m else "")
except Exception:
    print("")
PY
)"

# Map CUDA→PyTorch index (adjusts as Colab changes)
PT_IDX="https://download.pytorch.org/whl/cpu"
if [[ -n "${CUDA_VER}" ]]; then
  case "${CUDA_VER}" in
    12.4*) PT_IDX="https://download.pytorch.org/whl/cu124" ;;
    12.3*) PT_IDX="https://download.pytorch.org/whl/cu124" ;;  # 12.3 uses cu124 wheels
    12.1*) PT_IDX="https://download.pytorch.org/whl/cu121" ;;
    12.*)  PT_IDX="https://download.pytorch.org/whl/cu124" ;;   # fallback for newer 12.x
    11.*)  PT_IDX="https://download.pytorch.org/whl/cu118" ;;   # legacy fallbacks
  esac
fi

echo "Detected CUDA: '${CUDA_VER:-none}', using PyTorch index: ${PT_IDX}"

# 3) Install PyTorch (verbose)
python -m pip install --upgrade --verbose torch torchvision torchaudio --index-url "${PT_IDX}"

Detected CUDA: '12.4', using PyTorch index: https://download.pytorch.org/whl/cu124
Using pip 25.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Looking in indexes: https://download.pytorch.org/whl/cu124


In [ ]:
!python -m pip install --upgrade --verbose \
  numpy scipy pandas scikit-learn \
  matplotlib ipywidgets jupyterlab-widgets \
  tqdm rich loguru \
  pillow opencv-python-headless \
  einops pyyaml requests

Using pip 25.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Link requires a different Python (3.12.11 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/3a/be/650f9c091ef71cb01d735775d554e068752d3ff63d7943b26316dc401749/numpy-1.21.2.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.11 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/5f/d6/ad58ded26556eaeaa8c971e08b6466f17c4ac4d786cd3d800e26ce59cc01/numpy-1.21.3.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.11 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/fb/48/b0708ebd7718a8933f0d3937513ef8ef2f4f04529f1f66ca86d873043921/numpy-1.21.4.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.11 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/c2/a8/a924a09492bdfee8c2ec3094d0a1

In [ ]:
!pip install -U llama-cpp-python

## Local Inference on GPU
Model page: https://huggingface.co/mradermacher/Mistral-2x24B-MOE-Power-Magistral-Devstral-Reasoning-Ultimate-44B-i1-GGUF

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/mradermacher/Mistral-2x24B-MOE-Power-Magistral-Devstral-Reasoning-Ultimate-44B-i1-GGUF)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="mradermacher/Mistral-2x24B-MOE-Power-Magistral-Devstral-Reasoning-Ultimate-44B-i1-GGUF",
	filename="{{GGUF_FILE}}",
)


In [ ]:
llm.create_chat_completion(
	messages = "No input example has been defined for this model task."
)

# Task
Set up a Colab environment to run a DevStrill large language model using LLaMA-CPP-Python, including installing necessary system packages and over 50 Python libraries for machine learning, configuring GPU usage, implementing comprehensive logging, and creating a simple UI for interaction.

## System setup

### Subtask:
Install necessary system packages and configure the environment for GPU usage and logging.


**Reasoning**:
Check for available GPU and install any necessary system packages.



In [ ]:
!apt-get update
!apt-get install -y build-essential

In [5]:
!python -m pip install --upgrade --verbose pip setuptools wheel packaging build

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Obtaining dependency information for pip from https://files.pythonhosted.org/packages/b7/3f/945ef7ab14dc4f9d7f40288d2df998d1837ee0888ec3659c813487572faa/pip-25.2-py3-none-any.whl.metadata
  Obtaining dependency information for setuptools from https://files.pythonhosted.org/packages/a3/dc/17031897dae0efacfea57dfd3a82fdd2a2aeb58e0ff71b77b87e44edc772/setuptools-80.9.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Removing file or directory /usr/local/lib/python3.12/dist-packages/_distutils_hack/
      Removing file or directory /usr/local/lib/python3.12/dist-packages/distutils-precedence.pth
      Removing file or directory /usr/local/lib/python3.12/d